In [ ]:
!git clone https://github.com/vimalathithan17/quantum-classification-train.git
%cd /kaggle/working/quantum-classification-train

In [1]:
!pip install -q -r requirements.txt


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [ ]:
import os
import wandb

# Set environment variables
os.environ["WANDB_API_KEY"] = "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" # REPLACE THIS

# In this notebook, your inputs are the original split dataset AND the pretrained features dataset
os.environ["GLOBAL_TRAIN_DIR"] = "/kaggle/input/your-global-split-dataset/data/global_train" # REPLACE THIS
os.environ["GLOBAL_TEST_DIR"] = "/kaggle/input/your-global-split-dataset/data/global_test" # REPLACE THIS
os.environ["ENCODER_DIR"] = "/kaggle/input/your-global-split-dataset/master_label_encoder" # REPLACE THIS

# Where the previous contrastive output was saved
PRETRAINED_DIR = "/kaggle/input/your-pretrained-dataset/pretrained_features" # REPLACE THIS

os.environ["OUTPUT_DIR"] = "/kaggle/working/output"

# Ensure directories exist
!mkdir -p /kaggle/working/output

# Login to wandb
wandb.login(key=os.environ["WANDB_API_KEY"])

In [ ]:
# Copy previous run data if necessary (e.g., Optuna DB, pre-trained models)
# !cp -r /kaggle/input/previous-run-data/* /kaggle/working/output/
print("Setup complete. Ready to run.")

In [ ]:
# Tune hyperparameters using Optuna over 100% of global_train
!python tune_models.py \
    --datatype GeneExpr \
    --approach 1 \
    --qml_model reuploading \
    --use_pretrained_features \
    --pretrained_features_dir $PRETRAINED_DIR \
    --min_qbits 8 --max_qbits 14 \
    --min_layers 2 --max_layers 6 \
    --steps 75 \
    --n_trials 50 \
    --use_wandb --wandb_project qml-tuning --wandb_run_name member4_tuning_GeneExpr \
    --verbose

In [ ]:
# Train the final Base Learner model on 100% of global_train
# Generates OOF inferences to disk, then evaluates metrics entirely on global_test
!python dre_relupload.py \
    --datatypes GeneExpr \
    --use_pretrained_features \
    --pretrained_features_dir $PRETRAINED_DIR \
    --n_qbits 8 --n_layers 6 --steps 100 \
    --weight_decay 0.03 \
    --patience 25 \
    --validation_frac 0.2 \
    --use_wandb --wandb_project qml-classification --wandb_run_name member4_train_GeneExpr \
    --verbose